# Validation gate — parity, provenance and reproducibility

Recompute and compare the complete serving artifact. Check all six Delta tables against the expected synthetic pipeline when running in Workbench. Do not describe a local pass as a cloud validation. Keep notebook execution outputs out of Git.


In [ ]:
from pathlib import Path
import json, os, sys
root_hint = os.environ.get("AIDP_PROJECT_ROOT")
if root_hint:
    root = Path(root_hint).resolve()
else:
    root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "race_demo/pipeline.py").is_file()), None)
if root is None or not (root / "race_demo/pipeline.py").is_file():
    raise RuntimeError("Set AIDP_PROJECT_ROOT to this repository checkout.")
sys.path.insert(0, str(root))
from race_demo.pipeline import generate_bronze, build_silver, build_gold, serving_document, write_once, digest
from race_demo.workbench import publish_table, table_name
execution = os.environ.get("RACE_DEMO_EXECUTION", "LOCAL_DEMO")
if execution not in {"LOCAL_DEMO", "AIDP_WORKBENCH"}:
    raise ValueError("Unknown execution mode")
if execution == "AIDP_WORKBENCH":
    if "spark" not in globals():
        raise RuntimeError("Use an AIDP notebook runtime with Spark/Delta configured.")
    table_name("bronze")  # Validate the dedicated demo namespace before writes.
    if not os.environ.get("RACE_DEMO_OUTPUT"):
        raise RuntimeError("Set RACE_DEMO_OUTPUT to a new writable demo volume directory.")
output = Path(os.environ.get("RACE_DEMO_OUTPUT", str(root / "build/notebooks")))
print("Execution:", execution, "| Evidence: SYNTHETIC_DEMO")


In [ ]:
from race_demo.server import validate_serving
document = json.loads((output / "serving/demo.json").read_text())
if document["execution"] != execution:
    raise ValueError("Serving artifact comes from another execution mode")
validate_serving(document)
if execution == "AIDP_WORKBENCH":
    bronze = generate_bronze()
    silver = build_silver(bronze)
    gold = build_gold(silver)
    expected = {
        "bronze": bronze, "silver": silver, "gold_profiles": gold["profiles"],
        "gold_deviations": [{"sample_index": i, "deviation_seconds": d, "evidence_type": "SYNTHETIC_DEMO"} for i, d in enumerate(gold["deviations"])],
        "gold_scenarios": [{"field_size": s["field_size"], "simulations": s["simulations"], "probability": s["full_field_probability"], "seed": s["simulation_seed"], "evidence_type": "SYNTHETIC_DEMO"} for s in document["scenarios"].values()],
        "gold_curve": [{"field_size": s["field_size"], "delay_seconds": p["delaySeconds"], "probability": p["catchProbability"], "evidence_type": "SYNTHETIC_DEMO"} for s in document["scenarios"].values() for p in s["curve"]],
    }
    for layer, rows in expected.items():
        actual = [r.asDict() for r in spark.table(table_name(layer)).collect()]
        from race_demo.pipeline import canonical
        if digest(sorted(actual, key=canonical)) != digest(sorted(rows, key=canonical)):
            raise ValueError("Delta parity failed: " + layer)
print({"validation": "PASS", "execution": execution, "evidence_type": document["evidence_type"]})
